# Baseline retrieval evaluation ? DPqiIzK97K0

This notebook evaluates the fixed caption snapshot with the Phase 0 sequential word-window baseline. It runs independently of the product query path, builds a fresh video/config-specific local Chroma index, and uses the current hybrid dense + lexical reciprocal-rank fusion strategy.

**Hypothesis.** With 200 target words, 35 overlap words, 300 maximum words, the recorded embedding model, hybrid retrieval, and `k=5`, relevant evidence chunks will appear in the top five for a useful portion of the reviewed answerable questions. This is a baseline measurement, not a claim that these settings are best.

The retrieval code and local index live under `experimentation/`; no generation model or API key is used. Run all cells from top to bottom with the project `youtube-rag` kernel.

In [1]:
from __future__ import annotations
import csv, json, os, platform, sys, time
from dataclasses import asdict
from datetime import date, datetime, timezone
from pathlib import Path
import importlib.metadata as metadata
import pandas as pd
from IPython.display import display

ROOT = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "experimentation" / "evaluation" / "questions-v1.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Could not locate the repository root from the notebook working directory.")
os.chdir(ROOT)
EXP = ROOT / "experimentation"
ARTIFACTS = EXP / "artifacts" / "DPqiIzK97K0"
OUTPUTS = EXP / "outputs" / "baseline"
OUTPUTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EXP / "src"))
from baseline import RunConfig, HybridIndex, aggregate_metrics, load_jsonl, relevant_chunk_spans, score_question, sha256
from youtube_rag.chunking import SequentialWordChunker
from sentence_transformers import SentenceTransformer

d:\Projects\YoutubeRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Fixed inputs and run record

The caption hash guards against silent input changes. The evaluation file stores labels against original caption segment IDs and its own corpus hashes. No labels are passed to the index or retriever; they are used only after rankings are generated.

In [2]:
raw_path = ARTIFACTS / "raw" / "DPqiIzK97K0.json"
segments_path = ARTIFACTS / "processed" / "DPqiIzK97K0.segments.jsonl"
eval_path = EXP / "evaluation" / "questions-v1.json"
config = RunConfig(dense_candidate_depth=20)
raw_hash, segments_hash, eval_hash = map(sha256, [raw_path, segments_path, eval_path])
evaluation = json.loads(eval_path.read_text(encoding="utf-8"))
assert raw_hash == evaluation["corpus_raw_sha256"], "Raw corpus hash differs from reviewed evaluation data."
assert segments_hash == evaluation["validated_segments_sha256"], "Validated segment hash differs from reviewed evaluation data."
assert evaluation["video_id"] == config.video_id
segments = load_jsonl(segments_path)
questions = evaluation["questions"]

versions = {}
for package in ["sentence-transformers", "chromadb", "numpy", "pandas", "scikit-learn", "ipykernel"]:
    try: versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError: versions[package] = "unavailable"

run_date = date.today().isoformat()
git_revision = os.popen("git rev-parse HEAD").read().strip() or "unavailable"
run_record = {
    "run_id": None, "run_date": run_date, "git_revision": git_revision,
    "dataset_id": evaluation["dataset_id"], "evaluation_version": evaluation["version"],
    "video_id": config.video_id, "raw_corpus_sha256": raw_hash,
    "validated_segments_sha256": segments_hash, "evaluation_sha256": eval_hash,
    "notebook": "experimentation/notebooks/baseline-retrieval.ipynb",
    "configuration": asdict(config), "candidate_depth": config.dense_candidate_depth,
    "dependency_versions": versions, "python_version": platform.python_version(),
    "platform": platform.platform(), "hardware": platform.processor() or "unreported",
    "embedding_device": "sentence-transformers default device",
}
run_record

{'run_id': None,
 'run_date': '2026-10-19',
 'git_revision': '2f3ca4139137b021fa80387e813ff539b364814d',
 'dataset_id': 'DPqiIzK97K0-retrieval-eval-v1',
 'evaluation_version': '1.0.0',
 'video_id': 'DPqiIzK97K0',
 'raw_corpus_sha256': '9e3275e1fec88de4fb0440e51e53c6951cb85cd5b6bb7eb205cfde345dfd9952',
 'validated_segments_sha256': '5f02ff99f83854213c547d373a967d9bee4a718ac97de1c70cfb3f84c3a47e76',
 'evaluation_sha256': '4c18586597bc4743f4ad55a53248d6c738351298b5d0feefd58571e355e32184',
 'notebook': 'experimentation/notebooks/baseline-retrieval.ipynb',
 'configuration': {'video_id': 'DPqiIzK97K0',
  'target_words': 200,
  'overlap_words': 35,
  'max_words': 300,
  'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
  'k': 5,
  'dense_candidate_depth': 20,
  'rrf_constant': 60,
  'dense_rrf_weight': 1.0,
  'lexical_rrf_weight': 2.0,
  'strategy': 'hybrid_dense_lexical_rrf'},
 'candidate_depth': 20,
 'dependency_versions': {'sentence-transformers': '5.7.0',
  'chromadb': '1.5.9',

## Build baseline chunks and isolated index

Chunking settings match the recorded candidate. The model is loaded locally or downloaded by Sentence Transformers if absent. The Chroma index path is derived from the corpus and configuration hashes, stays inside this video's ignored artifact directory, and is rebuilt for each complete notebook run.

In [3]:
chunker = SequentialWordChunker(config.target_words, config.overlap_words, config.max_words)
chunk_objects, chunk_stats = chunker.chunk(segments, config.video_id)
chunks = [asdict(row) for row in chunk_objects]
chunk_words = [len(row["text"].split()) for row in chunks]
config_payload = json.dumps(asdict(config), sort_keys=True)
config_hash = __import__("hashlib").sha256((raw_hash + config_payload).encode()).hexdigest()[:12]
run_record["run_id"] = f"baseline-{config_hash}"
index_path = ARTIFACTS / "indexes" / run_record["run_id"]
model = SentenceTransformer(config.embedding_model)
index = HybridIndex(index_path, model, config)
indexing_seconds = index.build(chunks)
run_record["index_path"] = str(index_path.relative_to(ROOT))
run_record["indexing_seconds"] = indexing_seconds
run_record["chunk_stats"] = {
    "input_segments": len(segments), "chunk_count": len(chunks),
    "word_count_including_overlap": sum(chunk_words),
    "average_words": sum(chunk_words) / len(chunk_words),
    "maximum_words": max(chunk_words),
}
display(pd.DataFrame([run_record["chunk_stats"]]))
print(f"Built {index.collection.count()} indexed chunks in {indexing_seconds:.2f} seconds.")

Batches: 100%|██████████| 23/23 [00:25<00:00,  1.09s/it]


,input_segments,chunk_count,word_count_including_overlap,average_words,maximum_words
0,15502,725,147493,203.438621,210


Built 725 indexed chunks in 26.64 seconds.


## Retrieval and scoring

The retrieval candidate pool is the top 20 dense results (or the full collection if smaller) plus all nonzero lexical results. Reciprocal-rank fusion uses dense weight 1, lexical weight 2, and constant 60, matching the repository's current hybrid scorer. `k=5`.

Relevant chunks are derived per question from overlap with its labelled source segment IDs and evidence interval. Hit@5 is whether any relevant chunk appears in the top five. Recall@5 is the fraction of labelled evidence spans represented in the top five. MRR is the reciprocal rank of the first relevant chunk. Timestamp localization is the share of answerable questions with at least one retrieved chunk overlapping a labelled span. Unanswerable and ambiguous questions are retained in per-question output but excluded from these answerable-question metrics. These metrics evaluate evidence ranking, not answer confidence.

In [4]:
ranking_rows = []
retrieval_seconds = []
for question in questions:
    ranked, elapsed = index.retrieve(question["question"])
    retrieval_seconds.append(elapsed)
    score = score_question(question, ranked, chunks, config.k)
    score["retrieval_seconds"] = elapsed
    ranking_rows.append(score)

metrics = aggregate_metrics(ranking_rows)
answerable_times = [row["retrieval_seconds"] for row in ranking_rows if row["scored"]]
run_record["retrieval_latency_seconds"] = {
    "measurement_count": len(answerable_times),
    "median": float(pd.Series(answerable_times).median()),
    "p95": float(pd.Series(answerable_times).quantile(0.95)),
    "note": "rough single pass; includes query embedding and local retrieval",
}
run_record["metrics"] = metrics
display(pd.DataFrame([{**metrics, **run_record["retrieval_latency_seconds"]}]))
display(pd.DataFrame([{
    "question_id": row["question_id"], "answerability": row["answerability"],
    "hit_at_5": row["hit_at_k"], "recall_at_5": row["recall_at_k"],
    "reciprocal_rank": row["reciprocal_rank"],
    "timestamp_overlap_at_5": row["timestamp_overlap_at_k"],
    "top_chunk_ids": [r["chunk_id"] for r in row["rankings"]],
} for row in ranking_rows]))

,answerable_question_count,hit_at_k,recall_at_k,mrr,timestamp_overlap_rate_at_k,ambiguous_excluded,unanswerable_reported_separately,measurement_count,median,p95,note
0,21,0.761905,0.738095,0.623016,0.761905,1,3,21,0.07801,0.098039,rough single pass; includes query embedding an...


,question_id,answerability,hit_at_5,recall_at_5,reciprocal_rank,timestamp_overlap_at_5,top_chunk_ids
0,q01,answerable,True,1.0,1.000000,True,"[DPqiIzK97K0:0001, DPqiIzK97K0:0002, DPqiIzK97..."
1,q02,answerable,True,1.0,1.000000,True,"[DPqiIzK97K0:0008, DPqiIzK97K0:0033, DPqiIzK97..."
2,q03,answerable,True,1.0,1.000000,True,"[DPqiIzK97K0:0015, DPqiIzK97K0:0016, DPqiIzK97..."
3,q04,answerable,True,1.0,1.000000,True,"[DPqiIzK97K0:0022, DPqiIzK97K0:0301, DPqiIzK97..."
4,q05,answerable,True,1.0,0.333333,True,"[DPqiIzK97K0:0029, DPqiIzK97K0:0030, DPqiIzK97..."
5,q06,answerable,True,1.0,1.000000,True,"[DPqiIzK97K0:0047, DPqiIzK97K0:0706, DPqiIzK97..."
6,q07,answerable,True,1.0,0.250000,True,"[DPqiIzK97K0:0318, DPqiIzK97K0:0057, DPqiIzK97..."
7,q08,answerable,True,1.0,1.000000,True,"[DPqiIzK97K0:0091, DPqiIzK97K0:0334, DPqiIzK97..."
8,q09,answerable,True,1.0,1.000000,True,"[DPqiIzK97K0:0122, DPqiIzK97K0:0125, DPqiIzK97..."
9,q10,unanswerable,None,NaN,NaN,None,"[DPqiIzK97K0:0346, DPqiIzK97K0:0717, DPqiIzK97..."


## Per-question inspection

Review ranking regressions and the excerpts below alongside aggregate metrics. The label review notes that several questions refer to visually demonstrated components but labels were checked against the fixed auto-caption transcript. The dataset is one video and 25 questions, so these results do not establish statistical significance or general performance.

In [5]:
display(pd.DataFrame([{
    "question_id": row["question_id"], "question": row["question"],
    "answerability": row["answerability"], "scored": row["scored"],
    "hit_at_5": row["hit_at_k"], "recall_at_5": row["recall_at_k"],
    "rr": row["reciprocal_rank"],
    "top_results": "\n".join(f"{i}. {r['chunk_id']} [{r['start_sec']:.1f}-{r['end_sec']:.1f}s] {r['excerpt'][:260]}?" for i, r in enumerate(row["rankings"], 1)),
} for row in ranking_rows]))

,question_id,question,answerability,scored,hit_at_5,recall_at_5,rr,top_results
0,q01,Which alternative kit does Matthew recommend f...,answerable,True,True,1.0,1.000000,1. DPqiIzK97K0:0001 [55.7-128.1s] when I said ...
1,q02,Are terminal rows a through e connected across...,answerable,True,True,1.0,1.000000,1. DPqiIzK97K0:0008 [455.4-522.4s] horizontall...
2,q03,According to the instructor's calculations wit...,answerable,True,True,1.0,1.000000,1. DPqiIzK97K0:0015 [871.4-952.8s] limit of 20...
3,q04,What proportion of electrical energy in tradit...,answerable,True,True,1.0,1.000000,1. DPqiIzK97K0:0022 [1329.0-1415.2s] volt to t...
4,q05,Why does an N-type semiconductor acquire a net...,answerable,True,True,1.0,0.333333,1. DPqiIzK97K0:0029 [1775.8-1842.0s] of just u...
5,q06,How do void setup() and void loop() differ in ...,answerable,True,True,1.0,1.000000,1. DPqiIzK97K0:0047 [2756.6-2822.0s] it off it...
6,q07,"In the traffic light circuit implementation, w...",answerable,True,True,1.0,0.250000,1. DPqiIzK97K0:0318 [17423.4-17478.6s] obvious...
7,q08,How can a user visually identify PWM-capable d...,answerable,True,True,1.0,1.000000,1. DPqiIzK97K0:0091 [5085.0-5158.8s] all right...
8,q09,Why does the Arduino's 10-bit analogRead() ret...,answerable,True,True,1.0,1.000000,1. DPqiIzK97K0:0122 [6778.6-6848.2s] power of ...
9,q10,What is the clock frequency in megahertz of th...,unanswerable,False,None,NaN,NaN,1. DPqiIzK97K0:0346 [18861.7-18930.4s] okay so...


## Save portable run artifacts

The result JSON stores settings, environment, metrics, and the per-question rankings and metric inputs. The CSV is a compact inspection table. Generated outputs remain under the ignored local output directory; the notebook is the shareable narrative.

In [ ]:
portable = {
    "run_record": run_record,
    "metric_definitions": {
        "hit_at_k": "answerable-question fraction with >=1 relevant chunk in top k",
        "recall_at_k": "mean fraction of labelled evidence spans represented in top k",
        "mrr": "mean reciprocal rank of first relevant chunk over answerable questions",
        "timestamp_overlap_rate_at_k": "answerable-question fraction with a top-k chunk overlapping any labelled evidence interval",
        "interval_rule": "inclusive overlap: chunk.start_sec <= span.end_sec and chunk.end_sec >= span.start_sec",
    },
    "aggregate_metrics": metrics,
    "per_question": ranking_rows,
}
json_path = OUTPUTS / f"{run_record['run_id']}-results.json"
csv_path = OUTPUTS / f"{run_record['run_id']}-per-question.csv"
json_path.write_text(json.dumps(portable, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question_id", "answerability", "scored", "hit_at_k", "recall_at_k", "reciprocal_rank", "timestamp_overlap_at_k", "retrieval_seconds", "relevant_evidence_units", "evidence_units_found_at_k", "relevant_chunk_ids"])
    writer.writeheader()
    for row in ranking_rows: writer.writerow({key: row[key] for key in writer.fieldnames})
print(json_path.relative_to(ROOT))
print(csv_path.relative_to(ROOT))